In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
import neo4j
import os
import torch
from dotenv import load_dotenv
load_dotenv(override=True)
from neo4j import GraphDatabase
import json
import re
import torch
import pandas as pd

c:\Users\rompalag\AppData\Local\miniforge3\envs\bkb_rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
uri = os.environ["NEO4J_BOLT_URI"]
print(uri)
user = os.environ["NEO4J_USERNAME"]
print(user)
password = os.environ["NEO4J_PASSWORD"]
print(password)

neo4j://127.0.0.1:7687
neo4j
Omic1124hard!


In [ ]:
neokb_driver = neo4j.GraphDatabase.driver(uri, auth=(user, password))

class HuggingFaceEmbedder:
    def __init__(self, model_name="cambridgeltl/SapBERT-from-PubMedBERT-fulltext", device="cpu"):
        self.model = SentenceTransformer(model_name, device=device)
    def embed_query(self, text: str) -> list[float]:
        # returns the embedding as list of floats to match OpenAIEmbeddings
        vector = self.model.encode([text], convert_to_numpy=True, normalize_embeddings=True)[0]
        return vector.tolist()
    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        vectors = self.model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
        return vectors.tolist()

In [64]:
embedder = HuggingFaceEmbedder(device="cpu")
query_vector = embedder.embed_query("bladder")

No sentence-transformers model found with name cambridgeltl/SapBERT-from-PubMedBERT-fulltext. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 225.79it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: cambridgeltl/SapBERT-from-PubMedBERT-fulltext
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
from neo4j import GraphDatabase

def search_nodes(query_vector, k, neokb_driver):
    result = neokb_driver.execute_query(
        """
        CALL db.index.vector.queryNodes($index, $k, $embedding)
        YIELD node, score
        RETURN node.nodeId AS nodeId,
               node.name AS name,
               score
        ORDER BY score DESC
        """,
        parameters_={
            "index": "text_embeddings",
            "k": k,
            "embedding": query_vector,
        }
    )
    return [record.data() for record in result.records]


In [65]:
results = search_nodes(query_vector, k=10, neokb_driver=neokb_driver)

print(results)

[{'nodeId': 904, 'name': 'uterus', 'score': 0.7349241971969604}, {'nodeId': 945, 'name': 'thoracic mammary gland', 'score': 0.7010969519615173}, {'nodeId': 991, 'name': 'extracellular space', 'score': 0.6983031034469604}, {'nodeId': 912, 'name': 'fundus of stomach', 'score': 0.6976017951965332}, {'nodeId': 907, 'name': 'esophagus', 'score': 0.6960209012031555}, {'nodeId': 926, 'name': 'cerebellum', 'score': 0.6921419501304626}, {'nodeId': 928, 'name': 'cardiac ventricle', 'score': 0.6913538575172424}, {'nodeId': 929, 'name': 'heart left ventricle', 'score': 0.6878123879432678}, {'nodeId': 938, 'name': 'bone marrow', 'score': 0.6870373487472534}, {'nodeId': 942, 'name': 'trachea', 'score': 0.6863113641738892}]


In [59]:
import torch
pcst_output = torch.load("stark_qa_v0_0/processed/train_pcst_output.pt", weights_only=False)

In [ ]:
pcst_output.keys()

In [63]:
first_query_id = list(pcst_output['pcst_nodes'].keys())[0]
print("Query ID:", first_query_id)
first_pcst_nodes = pcst_output["pcst_nodes"][first_query_id]

print(first_pcst_nodes)

Query ID: 454
[482, 169, 561, 292, 419, 203, 178, 395, 89, 444, 543, 330, 499, 125, 514, 519, 469, 148, 345, 258, 103, 108, 416, 366, 288, 324, 147, 59, 87, 174, 75, 329, 489, 385, 208, 270, 62, 257, 267, 527, 235, 372, 340, 508, 717, 698, 765, 554, 511, 255, 213, 635, 505, 272, 406, 427, 775, 411, 390, 117, 53, 631, 701, 557, 171, 128, 686, 544, 249, 512, 691, 784, 480, 468, 301, 353, 819, 510, 402, 68, 710, 706, 533, 704, 724, 457, 719]


In [67]:
load_dotenv('.env', override=True)
NEO4J_BOLT_URI = os.getenv('NEO4J_BOLT_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

In [69]:
query_embedding_dict = torch.load('stark_qa_data/query_emb_dict.pt')

In [71]:
from sklearn.metrics.pairwise import cosine_similarity

In [73]:
ordered_pcst_nodes_20 = {}
with GraphDatabase.driver(NEO4J_BOLT_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
    for query_pos, pcst_nodes in pcst_output['pcst_nodes'].items():
        item = list(query_embedding_dict.items())[query_pos]
        query_id, temp_val = item
        query_emb = query_embedding_dict[query_id]
        res = driver.execute_query("""
        UNWIND $nodeIds AS nodeId
        MATCH (node:_Entity_ {nodeId:nodeId}) RETURN node.nodeId as nodeId, node.textEmbedding AS textEmbedding
        """, parameters_={"nodeIds": pcst_nodes})
        node_embs = pd.DataFrame([rec.data() for rec in res.records])
        embeddings = np.vstack(node_embs['textEmbedding'].values)
        cos_sim = cosine_similarity(embeddings, query_emb.reshape(1,-1)).ravel()
        top_n_indices = np.argsort(cos_sim)[-20:][::-1]
        top_n_nodeIds = node_embs.iloc[top_n_indices]['nodeId'].to_numpy()
        ordered_pcst_nodes_20[query_id] = top_n_nodeIds

In [75]:
ordered_pcst_nodes_20[4469]

array([482, 169, 561, 292, 419, 203, 178, 395,  89, 444, 543, 330, 499,
       125, 514, 519, 469, 148, 345, 258])